In [1]:
import zarr

import numpy as np

from scipy.ndimage import binary_dilation
from scipy.ndimage import label as ndi_label, binary_fill_holes
from scipy.ndimage import label as ndi_label
from scipy.ndimage import binary_fill_holes

import napari
from napari.utils.colormaps.standardize_color import transform_color

Prep: Input the cell IDs that we are going to reconstruct its 2D segmentation

In [2]:
cell_ids = [9456, 9597]
image_path = "./data"
saved_path = f'./candidates/{"_".join(map(str, cell_ids))}.npy'

In [3]:
zarr_image = zarr.open(image_path, mode="r")
segmentation = np.array(zarr_image)
highlighted_cells = np.zeros_like(segmentation, dtype=np.uint8)
for label, cell_id in enumerate(cell_ids, start=1):
    highlighted_cells[segmentation == cell_id] = label
viewer = napari.Viewer()
viewer.add_image(segmentation, name="Segmentation Image")
viewer.add_labels(highlighted_cells, name="Highlighted Cells")
napari.run()

Step 1: Finding the connecting curvature between cells

In [4]:
cell_1_mask = highlighted_cells == 1
cell_2_mask = highlighted_cells == 2
boundary_1 = binary_dilation(cell_1_mask) & cell_2_mask
boundary_2 = binary_dilation(cell_2_mask) & cell_1_mask
connecting_surface = boundary_1 | boundary_2
surface_coords = np.argwhere(connecting_surface)
curvature_layer = np.zeros_like(segmentation, dtype=np.uint8)

Optional: Visualizing the curvature

In [22]:
curvature_layer = np.zeros_like(segmentation, dtype=np.uint8)
for coord in surface_coords:
    x, y, z = coord
    curvature_layer[x, y, z] = 1
viewer = napari.Viewer()
viewer.add_image(segmentation, name="Segmentation")
viewer.add_labels(curvature_layer, name="Connecting Surface")
napari.run()

Step2: Fit the curvature via PCA plane

In [5]:
x = surface_coords[:, 0]
y = surface_coords[:, 1]
z = surface_coords[:, 2]

coords = surface_coords - surface_coords.mean(axis=0)
cov_matrix = np.cov(coords.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

normal_vector = eigenvectors[:, np.argmin(eigenvalues)]
plane_point = surface_coords.mean(axis=0)

x_range = np.arange(0, segmentation.shape[0])
y_range = np.arange(0, segmentation.shape[1])
yy, xx = np.meshgrid(y_range, x_range)

n1, n2, n3 = normal_vector
x0, y0, z0 = plane_point
zz = -(n1 * (xx - x0) + n2 * (yy - y0)) / n3 + z0

fitted_plane_layer = np.zeros_like(segmentation, dtype=np.uint8)

thickness = 1 
for z in range(segmentation.shape[2]):
    mask = (np.abs(zz - z) <= thickness)
    if np.any(mask): 
        fitted_plane_layer[xx[mask], yy[mask], z] = 2
highlighted_cells = np.zeros_like(segmentation, dtype=np.uint8)
for label, cell_id in enumerate(cell_ids, start=1):
    highlighted_cells[segmentation == cell_id] = label

viewer = napari.Viewer()
viewer.add_labels(fitted_plane_layer, name="PCA plane", opacity=0.9)
viewer.add_image(
    highlighted_cells == 1,
    name="Cell 1",
    colormap="pink",
    opacity=0.3,
    blending="additive"
)
viewer.add_image(
    highlighted_cells == 2,
    name="Cell 2",
    colormap="purple",
    opacity=0.3,
    blending="additive"
)
napari.run()

Step3: Using new plane to cut the cell and rotate the cell

In [6]:
cell_id_to_label = {cell_ids[0]: 1, cell_ids[1]: 2}
cell_mask = np.isin(segmentation, cell_ids)
cell_coords = np.argwhere(cell_mask)
projections = np.dot(cell_coords - plane_point, normal_vector)
min_proj = projections.min()
max_proj = projections.max()

padding = 5.0
step_size = 1.0  
tolerance = 0.5  
steps = np.arange(min_proj - padding, max_proj + padding, step_size)
if np.allclose(normal_vector, [0, 0, 1]) or np.allclose(normal_vector, [0, 0, -1]):
    u = np.array([1, 0, 0])
else:
    u = np.cross(normal_vector, [0, 0, 1])
    u /= np.linalg.norm(u)
v = np.cross(normal_vector, u)
v /= np.linalg.norm(v)
s_all = []
t_all = []
slice_indices = []
labels_all = []

for idx, step in enumerate(steps):
    shifted_plane_point = plane_point + step * normal_vector
    distances = np.dot(cell_coords - shifted_plane_point, normal_vector)
    mask = np.abs(distances) <= tolerance

    intersected_points = cell_coords[mask]

    if intersected_points.size == 0:
        continue
    p = intersected_points - shifted_plane_point
    s = np.dot(p, u)
    t = np.dot(p, v)

    s_all.extend(s)
    t_all.extend(t)
    slice_indices.extend([idx] * len(s))
    original_labels = segmentation[intersected_points[:, 0],
                                   intersected_points[:, 1],
                                   intersected_points[:, 2]]
    mapped_labels = [cell_id_to_label.get(label, 0) for label in original_labels]
    labels_all.extend(mapped_labels)
s_all = np.array(s_all)
t_all = np.array(t_all)
slice_indices = np.array(slice_indices)
labels_all = np.array(labels_all)
min_s = s_all.min()
max_s = s_all.max()
min_t = t_all.min()
max_t = t_all.max()
pixel_size = 1.0 

s_bins = int(np.ceil((max_s - min_s) / pixel_size)) + 1
t_bins = int(np.ceil((max_t - min_t) / pixel_size)) + 1

num_slices = len(steps)
image_stack = np.zeros((num_slices, s_bins, t_bins), dtype=np.uint8)
s_indices = np.floor((s_all - min_s) / pixel_size).astype(int)
t_indices = np.floor((t_all - min_t) / pixel_size).astype(int)
for s_idx, t_idx, slice_idx, label in zip(s_indices, t_indices, slice_indices, labels_all):
    image_stack[slice_idx, s_idx, t_idx] = label

for idx in range(image_stack.shape[0]):
    slice = image_stack[idx, :, :]
    for label_value in [1, 2]:
        label_mask = (slice == label_value)
        if np.any(label_mask):
            labeled_array, num_features = ndi_label(label_mask, structure=np.ones((3,3)))
            for component_idx in range(1, num_features + 1):
                component_mask = (labeled_array == component_idx)
                filled_component = binary_fill_holes(component_mask)
                slice[filled_component & ~component_mask] = label_value 
    image_stack[idx, :, :] = slice

label_colors = {
    0: 'black', 
    1: 'red',  
    2: 'green',
}
label_color_dict = {label: transform_color(color)[0] for label, color in label_colors.items()}
viewer = napari.Viewer()
layer = viewer.add_labels(image_stack, name="Reconstructed 2d segmentation using PCA plane")
layer.color = label_color_dict
napari.run()


Optional: checking how many gap layers there are 

In [9]:
cnt = 0
for idx in range(num_slices):
    slice = image_stack[idx, :, :]
    if np.any(slice == 1) and np.any(slice == 2):
        cnt += 1
print(cnt)

12


Step 4: create the gap and input to the pretrained model

In [17]:
normal_vector = normal_vector / np.linalg.norm(normal_vector)
cell_id_to_label = {cell_ids[0]: 1, cell_ids[1]: 2}
plane_layer_index = np.argmin(np.abs(steps))

plane_slice = image_stack[plane_layer_index, :, :]
area_i = np.sum(plane_slice > 0)

plane_slice = image_stack[plane_layer_index + 1, :, :]
area_j = np.sum(plane_slice > 0)

plane_slice = image_stack[plane_layer_index - 1, :, :]
area_k = np.sum(plane_slice > 0)

print(area_i, area_j, area_k)
image_stack[plane_layer_index, :, :] = 0

for idx in range(plane_layer_index + 1, num_slices):
    slice = image_stack[idx, :, :]
    slice[slice == 2] = 0
    image_stack[idx, :, :] = slice
for idx in range(0, plane_layer_index):
    slice = image_stack[idx, :, :]
    slice[slice == 1] = 0
    image_stack[idx, :, :] = slice

label_colors = {
    0: 'black',   
    1: 'blue',    
    2: 'yellow',  
}
label_color_dict = {label: transform_color(color)[0] for label, color in label_colors.items()}

viewer = napari.Viewer()
layer = viewer.add_labels(image_stack, name="Reconstructed 2d segmentation using PCA plane")
layer.color = label_color_dict
np.save(saved_path, image_stack)


0 928 605


In [1]:
import CandidateSearching as CS
import os
import re
import numpy as np
from cellstitch import interpolate as IP

from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
from scipy import ndimage
from cellstitch import evaluation as eval
import h5py


filepath = "./candidates"
files = os.listdir(filepath)
problem_masks = []
for k, filename in enumerate(files):
    print(f"Checking: {filename}")
    file_path = os.path.join(filepath, filename)
    result = CS.main(file_path)
    for j in result:
        problem_masks.append((filename, j[0], j[1], j[2], j[3]))
print("List of problematic masks in this dataset:")
for i in problem_masks:
    print(i)

Checking: 6982_6287.npy
0.9579169443247622
1 0.9579169443247622
[6.788910807069419e-05, 0.00010107446449063637, 0.00030580557131027695, 5.3764384622473426e-05, 0.0527530399923516, 5.025717015028758e-05, 6.090943647262364e-05, 7.161828728075137e-05, 4.389072276933392e-05, 0.010126974513893394, 6.0924463632675195e-05, -1.4144873148553383]
Checking: 7431_8630.npy
0.6002708898129621
1 0.6002708898129621
[4.8697663606877773e-05, 5.830578639448266e-05, 7.615365377014917e-05, 3.7865329933505596e-05, 0.007328330670983078, 9.664442873110692e-05, 0.00011918088760296382, 0.0002355777901595405, 8.414189410079956e-05, 0.02318731047859078, 0.00012402408879265365, -4.345293692946139]
Checking: 3123_3258.npy
0.809839877406346
1 0.809839877406346
[7.447467473197214e-05, 0.00020298907063777334, 0.000947227074485955, 5.685563298213858e-05, 0.056769895792742844, 1.5052058644182862e-05, 1.9949862503955892e-05, 4.3424439836814915e-05, 1.0233650868835267e-05, 0.017221953795100668, 5.398953019000482e-05, -2.6